# Final analysis - dashboard numbers and insights

Project: **Sentiment Analysis with Facebook**

Compute the same analytics the dashboard shows, directly from the database.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
from src import config
from src.data.load_data import load_raw, inspect_dataset
from src.data.preprocess import prepare_dataframe, clean_text
from backend.app.database.database import load_posts_df
from src.analysis import sentiment as sa, keywords as kw
from src.analysis.insights import generate_insights

In [2]:
posts = load_posts_df()
d = sa.distribution(posts)
print(d)
print('health:', sa.health_score(d['counts'], d['total']), '| voice index:', sa.voice_index(d['counts']))

{'total': 5000, 'counts': {'negative': 834, 'neutral': 1913, 'positive': 2253}, 'percentages': {'negative': 16.68, 'neutral': 38.26, 'positive': 45.06}, 'overall_sentiment': 'POSITIVE'}
health: 64.2 | voice index: 46.0


In [3]:
pd.DataFrame(kw.keyword_sentiment(posts, top_n=10, min_count=30))[['keyword','count','positive_pct','neutral_pct','negative_pct']]

,keyword,count,positive_pct,neutral_pct,negative_pct
0,new,717,39.3,44.4,16.3
1,day,359,47.1,33.7,19.2
2,update,258,23.6,50.8,25.6
3,coffee,233,41.2,38.2,20.6
4,maker,233,41.2,38.2,20.6
5,smartwatch,231,35.5,44.2,20.3
6,headphones,230,34.8,44.3,20.9
7,work,225,50.2,36.0,13.8
8,laptop,218,39.4,41.7,18.8
9,sneakers,216,38.0,45.8,16.2


In [4]:
ev = json.loads(config.EVALUATION_PATH.read_text(encoding='utf-8'))
rep = json.loads(config.DATA_REPORT_PATH.read_text(encoding='utf-8'))
for i in generate_insights(posts, ev, rep):
    print('-', i['text'])

- Overall sentiment is POSITIVE: 45.06% positive, 38.26% neutral and 16.68% negative across 5,000 posts.
- Negative sentiment is highest in 'customer_support' posts (62.83% negative vs 16.68% overall, 374 posts).
- 'meme_joke' posts are the most positive (68.62% positive, 564 posts).
- Negative sentiment is concentrated around the keyword 'please': 39.2% of the 181 posts that mention it are negative (overall: 16.68%).
- Posts mentioning 'friend' are the most positive: 73.7% of 114 posts.
- Negative sentiment is stable: 15.2% in the latest 90 days vs 16.8% earlier.
- 659 of 1,639 unique comment texts appear with more than one sentiment label, which caps text-only accuracy at about 70.08%.
- The selected model (Multinomial Naive Bayes) reaches macro-F1 0.51 and accuracy 0.54 on unseen test posts, versus macro-F1 0.21 for a majority-class baseline.
